In [1]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'Scalar.py', 'Encoder.py', 'Balancing_Data.py', 'Cross_Validation.py', 'CA1.csv', 'sample_data']


In [2]:
import sys
sys.path.append('/content')

In [3]:
import warnings
warnings.filterwarnings("ignore")

## **Evaluating Frequency of the Dataset**

In [4]:
import pandas as pd
# Evaluating Frequency of Dataset
df = pd.read_csv("CA1.csv")
print(df['Revenue'].value_counts())
print(df['Revenue'].value_counts(normalize=True) * 100)

Revenue
False    10422
True      1908
Name: count, dtype: int64
Revenue
False    84.525547
True     15.474453
Name: proportion, dtype: float64


# **Data is imbalance. Becasue the difference between False and True is too large. So, there is a need to balance the data Undersampling, Oversampling, and SMOTE.**


# **Checking the Missing Values**

In [5]:
# Check missing values
print("Missing Values in Each Column:")
print(df.isnull().sum())
# Percentage of missing values
print("\nMissing Values Percentage:")
print((df.isnull().sum() / len(df)) * 100)

Missing Values in Each Column:
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

Missing Values Percentage:
Administrative             0.0
Administrative_Duration    0.0
Informational              0.0
Informational_Duration     0.0
ProductRelated             0.0
ProductRelated_Duration    0.0
BounceRates                0.0
ExitRates                  0.0
PageValues                 0.0
SpecialDay                 0.0
Month                      0.0
OperatingSystems           0.0
Browser                    0.0
Reg

# **Checking the Outliers**

In [6]:
import numpy as np
numerical_cols = [
"Administrative",
"Administrative_Duration",
"Informational",
"Informational_Duration",
"ProductRelated",
"ProductRelated_Duration",
"BounceRates",
"ExitRates",
"PageValues",
"SpecialDay"
]
for col in numerical_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower = Q1 - 1.5 * IQR
  upper = Q3 + 1.5 * IQR
  outliers = df[(df[col] < lower) | (df[col] > upper)]
  print(col, "Outliers:", len(outliers))

Administrative Outliers: 404
Administrative_Duration Outliers: 1172
Informational Outliers: 2631
Informational_Duration Outliers: 2405
ProductRelated Outliers: 987
ProductRelated_Duration Outliers: 961
BounceRates Outliers: 1551
ExitRates Outliers: 1099
PageValues Outliers: 2730
SpecialDay Outliers: 1251


In [7]:
print("Original Shape:", df.shape)

Original Shape: (12330, 18)


# **Removal of Outliers**

In [8]:
numerical_cols = [
"Administrative",
"Administrative_Duration",
"Informational",
"Informational_Duration",
"ProductRelated",
"ProductRelated_Duration",
"BounceRates",
"ExitRates",
"PageValues",
"SpecialDay"
]

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

In [9]:
print("After Removing Outliers:", df.shape)

After Removing Outliers: (5016, 18)


## **PREPROCESSING**

In [ ]:
# ================================
#        PREPROCESSING
# ================================
from Encoder import Encoder
from Scalar import Scaler
from Balancing_Data import Balancing_Data
from Cross_Validation import CrossValidation
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
# Load Dataset
df = pd.read_csv("CA1.csv")
print("---------------------------------------------")
print("Original Dataset Shape:", df.shape)
print("---------------------------------------------")
# Separate Features & Target
X = df.drop("Revenue", axis=1)
y = df["Revenue"].astype(int)
# -------------------------------
#       Train Test Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y )
print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)
# -------------------------------
#          Encoding
# -------------------------------
binary_cols = ["Weekend"]
nominal_cols = ["Month", "VisitorType"]
encoder = Encoder()
# Fit on TRAIN
X_train = encoder.label_encode(X_train, binary_cols)
X_train = encoder.onehot_encode(X_train, nominal_cols)
# Transform TEST (no fitting again)
X_test = encoder.label_encode(X_test, binary_cols)
X_test = encoder.transform_onehot(X_test)
print("---------------------------------------------")
print("After Encoding - Train Shape:", X_train.shape)
print("After Encoding - Test Shape :", X_test.shape)
print("---------------------------------------------")
# -------------------------------
#        Scaling
# -------------------------------
numerical_cols = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay"
]
scaler = Scaler(method="standard")
# Fit on TRAIN
X_train = scaler.fit_transform(X_train, numerical_cols)
# Transform TEST
X_test = scaler.transform(X_test)
print("---------------------------------------------")
print("After Scaling - Train Shape:", X_train.shape)
print("After Scaling - Test Shape :", X_test.shape)
print("---------------------------------------------")

---------------------------------------------
Original Dataset Shape: (12330, 18)
---------------------------------------------
Train Shape: (9864, 17)
Test Shape : (2466, 17)
---------------------------------------------
After Encoding - Train Shape: (9864, 26)
After Encoding - Test Shape : (2466, 26)
---------------------------------------------
---------------------------------------------
After Scaling - Train Shape: (9864, 26)
After Scaling - Test Shape : (2466, 26)
---------------------------------------------


In [ ]:

balancer = Balancing_Data(X_train, y_train)
# =====================================================
#        Oversampling WITH Random State
# =====================================================
print("-----------------------------------")
print("\n Oversampling WITH Random State")
print("-----------------------------------")
X_over_rs, y_over_rs = balancer.Oversampling_with_RandomState(42)
# Logistic Regression
log_model = LogisticRegression()
log_model.fit(X_over_rs, y_over_rs)
y_pred = log_model.predict(X_test)
print("-----------------------------------")
print("Logistic Regression Results using Oversampling WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))
# Gaussian NB
gnb_model = GaussianNB()
gnb_model.fit(X_over_rs, y_over_rs)
y_pred = gnb_model.predict(X_test)
print("-----------------------------------")
print("Gaussian Naive Bayes Results using Oversampling WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))
# Decision Tree
dt_model = DecisionTreeClassifier()
dt_model.fit(X_over_rs, y_over_rs)
y_pred = dt_model.predict(X_test)
print("-----------------------------------")
print("Decision Tree Results using Oversampling WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))
# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_over_rs, y_over_rs)
y_pred = rf_model.predict(X_test)
print("-----------------------------------")
print("Random Forest Results using Oversampling WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))
# =====================================================
#        Oversampling WITHOUT Random State
# =====================================================
print("-----------------------------------")
print("\n Oversampling WITHOUT Random State")
print("-----------------------------------")
X_over, y_over = balancer.Oversampling_without_RandomState()
log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_over, y_over)
y_pred = log_model.predict(X_test)
print("-----------------------------------")
print("Logistic Regression Results using Oversampling WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

gnb_model = GaussianNB()
gnb_model.fit(X_over, y_over)
y_pred = gnb_model.predict(X_test)
print("-----------------------------------")
print("Gaussian Naive Bayes Results using Oversampling WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_over, y_over)
y_pred = dt_model.predict(X_test)
print("-----------------------------------")
print("Decision Tree Results using Oversampling WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_over, y_over)
y_pred = rf_model.predict(X_test)
print("-----------------------------------")
print("Random Forest Results using Oversampling WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))
# =====================================================
#         Undersampling WITH Random State
# =====================================================
print("-----------------------------------")
print("\n Undersampling WITH Random State")
print("-----------------------------------")
X_under_rs, y_under_rs = balancer.Undersampling_with_RandomState(42)
log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_under_rs, y_under_rs)
y_pred = log_model.predict(X_test)
print("-----------------------------------")
print("Logistic Regression Results using Undersampling WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

gnb_model = GaussianNB()
gnb_model.fit(X_under_rs, y_under_rs)
y_pred = gnb_model.predict(X_test)
print("-----------------------------------")
print("Gaussian Naive Bayes Results using Undersampling WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_under_rs, y_under_rs)
y_pred = dt_model.predict(X_test)
print("-----------------------------------")
print("Decision Tree Results using Undersampling WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_under_rs, y_under_rs)
y_pred = rf_model.predict(X_test)
print("-----------------------------------")
print("Random Forest Results using Undersampling WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))


# =====================================================
#         Undersampling WITHOUT Random State
# =====================================================

print("-----------------------------------")
print("\n Undersampling WITHOUT Random State")
print("-----------------------------------")

X_under, y_under = balancer.Undersampling_without_RandomState()

log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_under, y_under)
y_pred = log_model.predict(X_test)
print("-----------------------------------")
print("Logistic Regression Results using Undersampling WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

gnb_model = GaussianNB()
gnb_model.fit(X_under, y_under)
y_pred = gnb_model.predict(X_test)
print("-----------------------------------")
print("Gaussian Naive Bayes Results using Undersampling WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_under, y_under)
y_pred = dt_model.predict(X_test)
print("-----------------------------------")
print("Decision Tree Results using Undersampling WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_under, y_under)
y_pred = rf_model.predict(X_test)
print("-----------------------------------")
print("Random Forest Results using Undersampling WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# =====================================================
#   SMOTE WITH Random State without Cross Validation
# =====================================================

print("-----------------------------------")
print("\n SMOTE WITH Random State")
print("-----------------------------------")
X_smote_rs, y_smote_rs = balancer.SMOTE_with_RandomState(42)
# Logistic Regression
log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_smote_rs, y_smote_rs)
y_pred = log_model.predict(X_test)

print("-----------------------------------")
print("Logistic Regression Results using SMOTE WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))


# Gaussian NB
gnb_model = GaussianNB()
gnb_model.fit(X_smote_rs, y_smote_rs)
y_pred = gnb_model.predict(X_test)

print("-----------------------------------")
print("Gaussian Naive Bayes Results using SMOTE WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))


# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_smote_rs, y_smote_rs)
y_pred = dt_model.predict(X_test)

print("-----------------------------------")
print("Decision Tree Results using SMOTE WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))


# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_smote_rs, y_smote_rs)
y_pred = rf_model.predict(X_test)

print("-----------------------------------")
print("Random Forest Results using SMOTE WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# =====================================================
# SMOTE WITHOUT Random State without Cross Validation
# =====================================================
print("-----------------------------------")
print("\n SMOTE WITHOUT Random State without Cross Validation")
print("-----------------------------------")
X_smote, y_smote = balancer.SMOTE_without_RandomState()
# Logistic Regression
log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_smote, y_smote)
y_pred = log_model.predict(X_test)
print("-----------------------------------")
print("Logistic Regression Results using SMOTE WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Gaussian NB
gnb_model = GaussianNB()
gnb_model.fit(X_smote, y_smote)
y_pred = gnb_model.predict(X_test)
print("-----------------------------------")
print("Gaussian Naive Bayes Results using SMOTE WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))
# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_smote, y_smote)
y_pred = dt_model.predict(X_test)
print("-----------------------------------")
print("Decision Tree Results using SMOTE WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_smote, y_smote)
y_pred = rf_model.predict(X_test)
print("-----------------------------------")
print("Random Forest Results using SMOTE WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

-----------------------------------

 Oversampling WITH Random State
-----------------------------------
-----------------------------------
Logistic Regression Results using Oversampling WITH Random State without Cross Validation 
-----------------------------------
Confusion Matrix:
 [[1815  269]
 [  98  284]]
Accuracy : 0.85117599351176
Precision: 0.5135623869801085
Recall   : 0.743455497382199
F1 Score : 0.6074866310160428
-----------------------------------
Gaussian Naive Bayes Results using Oversampling WITH Random State without Cross Validation 
-----------------------------------
Confusion Matrix:
 [[1085  999]
 [  48  334]]
Accuracy : 0.5754257907542579
Precision: 0.25056264066016504
Recall   : 0.8743455497382199
F1 Score : 0.3895043731778426
-----------------------------------
Decision Tree Results using Oversampling WITH Random State without Cross Validation 
-----------------------------------
Confusion Matrix:
 [[1932  152]
 [ 172  210]]
Accuracy : 0.8686131386861314
Preci

## **With Cross Validation**

In [ ]:
# =====================================================
#      CROSS VALIDATION + BALANCING EXPERIMENTS
# =====================================================
# =====================================================
#          Oversampling WITH Random State
# =====================================================
print("-----------------------------------")
print("\n Oversampling WITH Random State")
print("-----------------------------------")
X_bal, y_bal = balancer.Oversampling_with_RandomState(42)
# Logistic Regression
log_model = LogisticRegression(max_iter=2000)
cv_log = CrossValidation(log_model, X_bal, y_bal)
scores_log = cv_log.FiveFold()
print("-----------------------------------")
print("Logistic Regression Results using Oversampling WITH Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_log)
print("Mean CV Accuracy:", np.mean(scores_log))
log_model.fit(X_bal, y_bal)
y_pred = log_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Gaussian NB
gnb_model = GaussianNB()
cv_gnb = CrossValidation(gnb_model, X_bal, y_bal)
scores_gnb = cv_gnb.FiveFold()
print("-----------------------------------")
print("Gaussian Naive Bayes Results using Oversampling WITH Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_gnb)
print("Mean CV Accuracy:", np.mean(scores_gnb))
gnb_model.fit(X_bal, y_bal)
y_pred = gnb_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
cv_dt = CrossValidation(dt_model, X_bal, y_bal)
scores_dt = cv_dt.FiveFold()
print("-----------------------------------")
print("Decision Tree Results using Oversampling WITH Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_dt)
print("Mean CV Accuracy:", np.mean(scores_dt))
dt_model.fit(X_bal, y_bal)
y_pred = dt_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
cv_rf = CrossValidation(rf_model, X_bal, y_bal)
scores_rf = cv_rf.FiveFold()
print("-----------------------------------")
print("Random Forest Results using Oversampling WITH Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_rf)
print("Mean CV Accuracy:", np.mean(scores_rf))
rf_model.fit(X_bal, y_bal)
y_pred = rf_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# =====================================================
#        Oversampling WITHOUT Random State
# =====================================================
print("-----------------------------------")
print("\n Oversampling WITHOUT Random State")
print("-----------------------------------")
X_bal, y_bal = balancer.Oversampling_without_RandomState()
# Logistic Regression
log_model = LogisticRegression(max_iter=2000)
cv_log = CrossValidation(log_model, X_bal, y_bal)
scores_log = cv_log.FiveFold()
print("-----------------------------------")
print("Logistic Regression Results using Oversampling WITHOUT Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_log)
print("Mean CV Accuracy:", np.mean(scores_log))
log_model.fit(X_bal, y_bal)
y_pred = log_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Gaussian NB
gnb_model = GaussianNB()
cv_gnb = CrossValidation(gnb_model, X_bal, y_bal)
scores_gnb = cv_gnb.FiveFold()
print("-----------------------------------")
print("Gaussian Naive Bayes Results using Oversampling WITHOUT Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_gnb)
print("Mean CV Accuracy:", np.mean(scores_gnb))
gnb_model.fit(X_bal, y_bal)
y_pred = gnb_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
cv_dt = CrossValidation(dt_model, X_bal, y_bal)
scores_dt = cv_dt.FiveFold()
print("-----------------------------------")
print("Decision Tree Results using Oversampling WITHOUT Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_dt)
print("Mean CV Accuracy:", np.mean(scores_dt))
dt_model.fit(X_bal, y_bal)
y_pred = dt_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
cv_rf = CrossValidation(rf_model, X_bal, y_bal)
scores_rf = cv_rf.FiveFold()
print("-----------------------------------")
print("Random Forest Results using Oversampling WITHOUT Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_rf)
print("Mean CV Accuracy:", np.mean(scores_rf))
rf_model.fit(X_bal, y_bal)
y_pred = rf_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# =====================================================
#              SMOTE WITH Random State
# =====================================================
print("-----------------------------------")
print("\n SMOTE WITH Random State")
print("-----------------------------------")
X_bal, y_bal = balancer.SMOTE_with_RandomState(42)
# Logistic Regression
log_model = LogisticRegression(max_iter=2000)
cv_log = CrossValidation(log_model, X_bal, y_bal)
scores_log = cv_log.FiveFold()
print("-----------------------------------")
print("Logistic Regression Results using SMOTE WITH Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_log)
print("Mean CV Accuracy:", np.mean(scores_log))
log_model.fit(X_bal, y_bal)
y_pred = log_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Gaussian NB
gnb_model = GaussianNB()
cv_gnb = CrossValidation(gnb_model, X_bal, y_bal)
scores_gnb = cv_gnb.FiveFold()
print("-----------------------------------")
print("Gaussian Naive Bayes Results using SMOTE WITH Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_gnb)
print("Mean CV Accuracy:", np.mean(scores_gnb))
gnb_model.fit(X_bal, y_bal)
y_pred = gnb_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
cv_dt = CrossValidation(dt_model, X_bal, y_bal)
scores_dt = cv_dt.FiveFold()
print("-----------------------------------")
print("Decision Tree Results using SMOTE WITH Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_dt)
print("Mean CV Accuracy:", np.mean(scores_dt))
dt_model.fit(X_bal, y_bal)
y_pred = dt_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
cv_rf = CrossValidation(rf_model, X_bal, y_bal)
scores_rf = cv_rf.FiveFold()

print("-----------------------------------")
print("Random Forest Results using SMOTE WITH Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_rf)
print("Mean CV Accuracy:", np.mean(scores_rf))
rf_model.fit(X_bal, y_bal)
y_pred = rf_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# =====================================================
# SMOTE WITHOUT Random State
# =====================================================
print("-----------------------------------")
print("\n SMOTE WITHOUT Random State")
print("-----------------------------------")
X_bal, y_bal = balancer.SMOTE_without_RandomState()

# Logistic Regression
log_model = LogisticRegression(max_iter=2000)
cv_log = CrossValidation(log_model, X_bal, y_bal)
scores_log = cv_log.FiveFold()
print("-----------------------------------")
print("Logistic Regression Results using SMOTE WITHOUT Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_log)
print("Mean CV Accuracy:", np.mean(scores_log))
log_model.fit(X_bal, y_bal)
y_pred = log_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Gaussian NB
gnb_model = GaussianNB()
cv_gnb = CrossValidation(gnb_model, X_bal, y_bal)
scores_gnb = cv_gnb.FiveFold()
print("-----------------------------------")
print("Gaussian Naive Bayes Results using SMOTE WITHOUT Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_gnb)
print("Mean CV Accuracy:", np.mean(scores_gnb))
gnb_model.fit(X_bal, y_bal)
y_pred = gnb_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
cv_dt = CrossValidation(dt_model, X_bal, y_bal)
scores_dt = cv_dt.FiveFold()
print("-----------------------------------")
print("Decision Tree Results using SMOTE WITHOUT Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_dt)
print("Mean CV Accuracy:", np.mean(scores_dt))
dt_model.fit(X_bal, y_bal)
y_pred = dt_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
cv_rf = CrossValidation(rf_model, X_bal, y_bal)
scores_rf = cv_rf.FiveFold()
print("-----------------------------------")
print("Random Forest Results using SMOTE WITHOUT Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_rf)
print("Mean CV Accuracy:", np.mean(scores_rf))
rf_model.fit(X_bal, y_bal)
y_pred = rf_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))



-----------------------------------

 Oversampling WITH Random State
-----------------------------------
-----------------------------------
Logistic Regression Results using Oversampling WITH Random State WITH 5-Fold Cross Validation
-----------------------------------
5-Fold CV Scores: [0.82254197 0.8137931  0.82458771 0.8191904  0.82188906]
Mean CV Accuracy: 0.8204004472583852
Confusion Matrix:
 [[1816  268]
 [  98  284]]
Accuracy : 0.851581508515815
Precision: 0.5144927536231884
Recall   : 0.743455497382199
F1 Score : 0.6081370449678801
-----------------------------------
Gaussian Naive Bayes Results using Oversampling WITH Random State WITH 5-Fold Cross Validation
-----------------------------------
5-Fold CV Scores: [0.72901679 0.71244378 0.72233883 0.71154423 0.73193403]
Mean CV Accuracy: 0.7214555312271922
Confusion Matrix:
 [[1085  999]
 [  48  334]]
Accuracy : 0.5754257907542579
Precision: 0.25056264066016504
Recall   : 0.8743455497382199
F1 Score : 0.3895043731778426
-------

# **Training One Model using Minmax as per Discussion in the Presentation on March 10 2026**

In [11]:
from Encoder import Encoder
from Scalar import Scaler
from Balancing_Data import Balancing_Data
from Cross_Validation import CrossValidation
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
# Load Dataset
df = pd.read_csv("CA1.csv")
print("---------------------------------------------")
print("Original Dataset Shape:", df.shape)
print("---------------------------------------------")
# Separate Features & Target
X = df.drop("Revenue", axis=1)
y = df["Revenue"].astype(int)
# -------------------------------
#       Train Test Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y )
print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)
# -------------------------------
#          Encoding
# -------------------------------
binary_cols = ["Weekend"]
nominal_cols = ["Month", "VisitorType"]
encoder = Encoder()
# Fit on TRAIN
X_train = encoder.label_encode(X_train, binary_cols)
X_train = encoder.onehot_encode(X_train, nominal_cols)
# Transform TEST (no fitting again)
X_test = encoder.label_encode(X_test, binary_cols)
X_test = encoder.transform_onehot(X_test)
print("---------------------------------------------")
print("After Encoding - Train Shape:", X_train.shape)
print("After Encoding - Test Shape :", X_test.shape)
print("---------------------------------------------")
# -----------------------------------------------
#        Scaling the dataset using MinMax
# -----------------------------------------------
numerical_cols = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay"
]
scaler = Scaler(method="minmax")
# Fit on TRAIN
X_train = scaler.fit_transform(X_train, numerical_cols)
# Transform TEST
X_test = scaler.transform(X_test)
print("---------------------------------------------")
print("After Scaling - Train Shape:", X_train.shape)
print("After Scaling - Test Shape :", X_test.shape)
print("---------------------------------------------")

---------------------------------------------
Original Dataset Shape: (12330, 18)
---------------------------------------------
Train Shape: (9864, 17)
Test Shape : (2466, 17)
---------------------------------------------
After Encoding - Train Shape: (9864, 26)
After Encoding - Test Shape : (2466, 26)
---------------------------------------------
---------------------------------------------
After Scaling - Train Shape: (9864, 26)
After Scaling - Test Shape : (2466, 26)
---------------------------------------------


In [13]:
balancer = Balancing_Data(X_train, y_train)
# =====================================================
#        Oversampling WITH Random State
# =====================================================
print("-----------------------------------")
print("\n Oversampling WITH Random State")
print("-----------------------------------")
X_over_rs, y_over_rs = balancer.Oversampling_with_RandomState(42)
# Logistic Regression
log_model = LogisticRegression()
log_model.fit(X_over_rs, y_over_rs)
y_pred = log_model.predict(X_test)
print("-----------------------------------")
print("Logistic Regression Results using Oversampling WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))


# =====================================================
#   SMOTE WITH Random State without Cross Validation
# =====================================================

print("-----------------------------------")
print("\n SMOTE WITH Random State")
print("-----------------------------------")
X_smote_rs, y_smote_rs = balancer.SMOTE_with_RandomState(42)
# Logistic Regression
log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_smote_rs, y_smote_rs)
y_pred = log_model.predict(X_test)

print("-----------------------------------")
print("Logistic Regression Results using SMOTE WITH Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

# =====================================================
#         Undersampling WITHOUT Random State
# =====================================================

print("-----------------------------------")
print("\n Undersampling WITHOUT Random State")
print("-----------------------------------")

X_under, y_under = balancer.Undersampling_without_RandomState()

log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_under, y_under)
y_pred = log_model.predict(X_test)
print("-----------------------------------")
print("Logistic Regression Results using Undersampling WITHOUT Random State without Cross Validation ")
print("-----------------------------------")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

-----------------------------------

 Oversampling WITH Random State
-----------------------------------
-----------------------------------
Logistic Regression Results using Oversampling WITH Random State without Cross Validation 
-----------------------------------
Confusion Matrix:
 [[1730  354]
 [  95  287]]
Accuracy : 0.8179237631792377
Precision: 0.44773790951638065
Recall   : 0.7513089005235603
F1 Score : 0.5610948191593352
-----------------------------------

 SMOTE WITH Random State
-----------------------------------
-----------------------------------
Logistic Regression Results using SMOTE WITH Random State without Cross Validation 
-----------------------------------
Confusion Matrix:
 [[1769  315]
 [ 104  278]]
Accuracy : 0.8300892133008921
Precision: 0.4688026981450253
Recall   : 0.7277486910994765
F1 Score : 0.5702564102564103
-----------------------------------

 Undersampling WITHOUT Random State
-----------------------------------
-----------------------------------


In [16]:
# =====================================================
#      CROSS VALIDATION + BALANCING EXPERIMENTS
# =====================================================
# =====================================================
#          Oversampling WITH Random State
# =====================================================
print("-----------------------------------")
print("\n Oversampling WITH Random State")
print("-----------------------------------")
X_bal, y_bal = balancer.Oversampling_with_RandomState(42)
# Logistic Regression
log_model = LogisticRegression(max_iter=2000)
cv_log = CrossValidation(log_model, X_bal, y_bal)
scores_log = cv_log.FiveFold()
print("-----------------------------------")
print("Logistic Regression Results using Oversampling WITH Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_log)
print("Mean CV Accuracy:", np.mean(scores_log))
log_model.fit(X_bal, y_bal)
y_pred = log_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))


# =====================================================
#              SMOTE WITH Random State
# =====================================================
print("-----------------------------------")
print("\n SMOTE WITH Random State")
print("-----------------------------------")
X_bal, y_bal = balancer.SMOTE_with_RandomState(42)
# Logistic Regression
log_model = LogisticRegression(max_iter=2000)
cv_log = CrossValidation(log_model, X_bal, y_bal)
scores_log = cv_log.FiveFold()
print("-----------------------------------")
print("Logistic Regression Results using SMOTE WITH Random State WITH 5-Fold Cross Validation")
print("-----------------------------------")
print("5-Fold CV Scores:", scores_log)
print("Mean CV Accuracy:", np.mean(scores_log))
log_model.fit(X_bal, y_bal)
y_pred = log_model.predict(X_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))




-----------------------------------

 Oversampling WITH Random State
-----------------------------------
-----------------------------------
Logistic Regression Results using Oversampling WITH Random State WITH 5-Fold Cross Validation
-----------------------------------
5-Fold CV Scores: [0.80425659 0.7976012  0.80689655 0.79550225 0.7994003 ]
Mean CV Accuracy: 0.8007313789148591
Confusion Matrix:
 [[1736  348]
 [  94  288]]
Accuracy : 0.8207623682076237
Precision: 0.4528301886792453
Recall   : 0.7539267015706806
F1 Score : 0.5658153241650294
-----------------------------------

 SMOTE WITH Random State
-----------------------------------
-----------------------------------
Logistic Regression Results using SMOTE WITH Random State WITH 5-Fold Cross Validation
-----------------------------------
5-Fold CV Scores: [0.81235012 0.83898051 0.84407796 0.83778111 0.83898051]
Mean CV Accuracy: 0.8344340419718199
Confusion Matrix:
 [[1769  315]
 [ 104  278]]
Accuracy : 0.8300892133008921
Precis